[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C74_Gaussian_Splatting_Course/05_appearance_dynamic/05_appearance_dynamic.ipynb)

# C74 · 模块 05 · 外观、动态与边界

本 notebook 把讲解页的六个结论跑出来：

1. **3DGS 实际用的实球谐基**（deg 0–3，共 16 个），并验证它们在球面上正交归一；
2. 阶数 → 内存：**deg 3 时 SH 占每个高斯 81% 的存储**；
3. **二分求出每个阶数能表示的最窄高光波瓣** —— 实测 $\approx 125^\circ/(\ell{+}1)$，
   deg 3 只到 **31.2°**，而抛光塑料要 12.3°；
4. 压缩账：两类手段可乘，组合后是基线的 **0.132×**；
5. **浮物检测**：构造一个「训练视角完全看不见、留出视角一眼就看到」的浮物；
6. 4DGS 内存账：**逐帧独立 300 帧 70.8 GB vs 3 阶轨迹 0.368 GB（192×）**。

只用 numpy，CPU，离线。

In [ ]:
import numpy as np
print('numpy', np.__version__)

# 3DGS 官方的实球谐系数（utils/sh_utils.py）
C0 = 0.28209479177387814
C1 = 0.4886025119029199
C2 = np.array([1.0925484305920792, -1.0925484305920792, 0.31539156525252005,
               -1.0925484305920792, 0.5462742152960396])
C3 = np.array([-0.5900435899266435, 2.890611442640554, -0.4570457994644658,
               0.3731763325901154, -0.4570457994644658, 1.445305721320277,
               -0.5900435899266435])

def sh_basis(dirs, deg=3):
    '''返回 (N, (deg+1)²) 的实球谐基值。dirs 是 (N,3) 的单位向量。'''
    d = np.asarray(dirs, float)
    d = d / np.linalg.norm(d, axis=-1, keepdims=True)
    x, y, z = d[:, 0], d[:, 1], d[:, 2]
    out = [np.full(len(d), C0)]
    if deg >= 1:
        out += [-C1*y, C1*z, -C1*x]
    if deg >= 2:
        xx, yy, zz = x*x, y*y, z*z
        out += [C2[0]*x*y, C2[1]*y*z, C2[2]*(2*zz - xx - yy),
                C2[3]*x*z, C2[4]*(xx - yy)]
    if deg >= 3:
        xx, yy, zz = x*x, y*y, z*z
        out += [C3[0]*y*(3*xx - yy), C3[1]*x*y*z, C3[2]*y*(4*zz - xx - yy),
                C3[3]*z*(2*zz - 3*xx - 3*yy), C3[4]*x*(4*zz - xx - yy),
                C3[5]*z*(xx - yy), C3[6]*x*(xx - 3*yy)]
    return np.stack(out, 1)

for deg in [0, 1, 2, 3]:
    print(f'  deg {deg}: {(deg+1)**2:2d} 个基函数（每个颜色通道），'
          f'共 {3*(deg+1)**2:2d} 个系数')
assert sh_basis(np.array([[0, 0, 1.]]), 3).shape == (1, 16)

## 1 · 球谐基的正交归一性

$\int_{S^2} Y_i Y_j \,\mathrm d\omega = \delta_{ij}$。
用球面上的均匀采样（Fibonacci 球）做数值积分来验证 —— 这也顺便说明
「为什么低阶系数决定整体亮度」：$Y_{00}$ 是常数。

In [ ]:
def fibonacci_sphere(n):
    '''球面上近似均匀的 n 个方向。'''
    i = np.arange(n) + 0.5
    phi = np.arccos(1 - 2*i/n)
    theta = np.pi * (1 + 5**0.5) * i
    return np.stack([np.cos(theta)*np.sin(phi),
                     np.sin(theta)*np.sin(phi),
                     np.cos(phi)], 1)

D = fibonacci_sphere(200_000)
Y = sh_basis(D, 3)
# 数值积分：(4π/n) Σ Y_i Y_j 应为 δ_ij
G = (4*np.pi/len(D)) * (Y.T @ Y)
off = np.abs(G - np.eye(16))
print(f'Gram 矩阵与单位阵的最大偏差 {off.max():.2e}')
print(f'对角线范围 {np.diag(G).min():.6f} ~ {np.diag(G).max():.6f}')
print(f'非对角线最大绝对值 {np.abs(G - np.diag(np.diag(G))).max():.2e}')
assert off.max() < 5e-3, f'必须近似正交归一，实测偏差 {off.max():.2e}'
print('\n✓ 16 个基函数在球面上正交归一（数值积分误差 %.1e）' % off.max())

# Y_00 是常数 -> deg 0 的高斯在所有方向上同色
y00 = sh_basis(D, 0)[:, 0]
assert np.allclose(y00, C0), 'Y_00 必须是常数'
print(f'✓ Y_00 ≡ {C0:.10f}（常数）—— 所以 deg 0 的高斯各方向同色')
print(f'  而 3DGS 的求值最后加 0.5：color = Σ k·Y + 0.5，然后 clamp 到 [0,1]')
print(f'  所以「k_00 = 0」对应的颜色是 0.5 灰，而不是黑')

# 一个具体的方向依赖颜色
rng = np.random.default_rng(0)
k = np.zeros((16, 3)); k[0] = [1.0, 0.6, 0.2]/np.array([C0, C0, C0])*0.3
k[2] = [0.8, 0.0, -0.8]        # 沿 z 的线性梯度（Y_10 ∝ z）
def eval_color(k, dirs, deg=3):
    return np.clip(sh_basis(dirs, deg) @ k + 0.5, 0, 1)
print('\n一个带 deg1 梯度的高斯，从不同方向看：')
for name, d in [('+z', [0,0,1.]), ('-z', [0,0,-1.]), ('+x', [1.,0,0]), ('+y', [0,1.,0])]:
    c = eval_color(k, np.array([d]))[0]
    print(f'  从 {name:2s} 看: RGB {np.round(c, 4)}')
c_pz = eval_color(k, np.array([[0,0,1.]]))[0]
c_nz = eval_color(k, np.array([[0,0,-1.]]))[0]
assert c_pz[0] > c_nz[0] and c_pz[2] < c_nz[2], 'Y_10 ∝ z 应造成沿 z 的颜色梯度'
print('  ✓ 红蓝沿 z 反向变化 —— 这就是「视角依赖颜色」，而它与光源、法向都无关')

## 2 · 阶数、内存与那个 81%

In [ ]:
GEO = 3 + 3 + 4 + 1        # mu + scale + quat + alpha

def floats_per_gaussian(deg):
    return GEO + 3*(deg+1)**2

print(' deg  SH系数  每高斯floats  100万高斯   SH占比   180/(deg+1)')
for deg in [0, 1, 2, 3, 4]:
    nsh = 3*(deg+1)**2
    nf = floats_per_gaussian(deg)
    print(f'  {deg}    {nsh:3d}      {nf:3d}       {nf*4*1e6/1e9:.3f} GB   '
          f'{nsh/nf:5.0%}      {180/(deg+1):5.1f}°')

n0, n3 = floats_per_gaussian(0), floats_per_gaussian(3)
assert n0 == 14 and n3 == 59
assert abs(n3*4*1e6/1e9 - 0.236) < 1e-3
assert abs(3*16/n3 - 0.8136) < 1e-3
print(f'\n✓ deg 3: 59 floats，SH 占 {3*16/n3:.1%}，几何只占 {GEO/n3:.1%}')
print(f'✓ deg 3 是 deg 0 的 {n3/n0:.2f}× 内存')
print('  「3D 高斯溅泼」听起来是个几何方法，但它的内存瓶颈在**外观**')

## 3 · SH 能表示多窄的高光：把边界二分出来

常见说法是「角分辨率 $\approx 180^\circ/(\ell+1)$」。用 Phong 型高光
$\max(\cos\theta,0)^n$ 投影到带限球谐，二分求出「相对 L2 残差 20%」对应的最窄波瓣。

In [ ]:
TH = np.linspace(0, np.pi, 4001)
CT = np.cos(TH)
WQ = np.sin(TH)                        # 球面测度（轴对称，只需 sinθ dθ）

def zonal_fit_residual(phong_n, deg):
    '''把 max(cosθ,0)^n 投影到 degree<=deg 的带限（zonal）球谐，返回相对 L2 残差。'''
    tgt = np.maximum(CT, 0.0)**phong_n
    A = np.stack([np.polynomial.legendre.legval(CT, [0]*k + [1])
                  for k in range(deg+1)], 1)
    sw = np.sqrt(WQ)
    coef = np.linalg.lstsq(A*sw[:, None], tgt*sw, rcond=None)[0]
    fit = A @ coef
    return float(np.sqrt(np.sum(WQ*(fit-tgt)**2) / np.sum(WQ*tgt**2)))

def half_angle(phong_n):
    '''Phong 指数 n 对应的半强度半角（度）。'''
    return float(np.degrees(np.arccos(0.5**(1.0/phong_n))))

print('先看 deg 3 的残差随波瓣宽度怎么变：')
print(' 半强度半角   Phong n    deg3 残差   deg8 残差')
for n in [1, 2, 3, 5, 8, 12, 20, 50, 200]:
    print(f'   {half_angle(n):5.1f}°      {n:3d}       '
          f'{zonal_fit_residual(n,3):.3f}      {zonal_fit_residual(n,8):.3f}')

# 一个反直觉的细节：deg3 的最佳拟合在 n=2，不在 n=1
r1, r2, r3 = (zonal_fit_residual(n, 3) for n in [1, 2, 3])
assert r2 < r1 and r2 < r3, f'n=2 应是 deg3 的最佳拟合：{r1:.3f}/{r2:.3f}/{r3:.3f}'
print(f'\n⚠ deg 3 的最佳拟合在 n=2（残差 {r2:.3f}），而不是最宽的 n=1（{r1:.3f}）')
print('  因为 max(cosθ,0)² 本身就接近一个二次多项式，恰好落在 deg3 的基里。')
print('  推论：「角分辨率」是一个粗糙的概念 —— SH 是多项式基而不是低通滤波器，')
print('        有些形状恰好落在基里（拟合极好），有些不落在（拟合很差）。')

In [ ]:
def narrowest_lobe(deg, tol=0.20):
    '''二分求出 degree<=deg 能拟合到 tol 相对残差的最窄波瓣（返回半强度半角，度）。'''
    lo, hi = 1.0, 5000.0                # n 越大波瓣越窄
    if zonal_fit_residual(lo, deg) >= tol:
        return half_angle(lo)           # 连最宽的都拟合不了
    for _ in range(60):
        mid = (lo + hi)/2
        if zonal_fit_residual(mid, deg) < tol:
            lo = mid
        else:
            hi = mid
    return half_angle(lo)

print(' SH 阶   180/(deg+1) 的说法   实测 20% 残差界   实测/说法   125/(deg+1)')
meas = {}
for deg in [0, 1, 2, 3, 4, 8]:
    h = narrowest_lobe(deg); meas[deg] = h
    print(f'   {deg}         {180/(deg+1):6.1f}°           {h:6.1f}°        '
          f'{h/(180/(deg+1)):.2f}       {125/(deg+1):6.1f}°')

# 结论 1：实测值系统性地低于 180/(deg+1)
for deg in [2, 3, 4, 8]:
    assert meas[deg] < 180/(deg+1), f'deg {deg} 的实测界必须低于说法值'
    assert abs(meas[deg] - 125/(deg+1)) < 3.0, \
        f'deg {deg}: 实测 {meas[deg]:.1f}° 应接近 125/(deg+1)={125/(deg+1):.1f}°'
assert abs(meas[3] - 31.2) < 0.5, f'deg 3 应为 31.2°，实测 {meas[3]:.1f}°'
print(f'\n✓ 「180/(deg+1)」系统性地**高估**了 SH 的能力（实测只有它的 0.68–0.72 倍）')
print(f'✓ 更好的经验式是 125/(deg+1)：deg 2/3/4/8 分别给 '
      f'{125/3:.1f}/{125/4:.1f}/{125/5:.1f}/{125/9:.1f}°，'
      f'实测 {meas[2]:.1f}/{meas[3]:.1f}/{meas[4]:.1f}/{meas[8]:.1f}°')
print('  （注意这个常数依赖「20% 残差」这个判据；换成 10% 或 30% 常数会变）')

In [ ]:
# 对照真实材质
print('材质            Phong n   高光半角    deg3 残差   deg3 能表示吗')
MATS = [('粗糙塑料', 5), ('抛光塑料', 30), ('漆面/清漆', 80), ('抛光金属', 400)]
for name, n in MATS:
    h = half_angle(n); r = zonal_fit_residual(n, 3)
    ok = '勉强' if r < 0.30 else ('不能' if r < 0.9 else '完全不能')
    print(f'  {name:12s}   {n:4d}    {h:5.1f}°      {r:.3f}      {ok}')

assert zonal_fit_residual(5, 3) < 0.30, '粗糙塑料 deg3 勉强可以'
assert zonal_fit_residual(30, 3) > 0.60, '抛光塑料 deg3 必须明显失败'
assert zonal_fit_residual(400, 3) > 0.95, '抛光金属 deg3 必须完全失败'
print(f'\n✓ SH deg 3 只能勉强表示粗糙塑料级别的高光，更亮的材质都表示不了')
print('  这是**表示能力**的限制，不是优化没收敛 —— 再训练一百万步也不会好')

# 提高阶数救不了：算一下代价
need = 30
deg_need = int(np.ceil(125/half_angle(need) - 1))
print(f'\n要表示抛光塑料（半角 {half_angle(need):.1f}°）需要 deg ≈ '
      f'125/{half_angle(need):.1f} - 1 = {deg_need}')
nf_need = GEO + 3*(deg_need+1)**2
print(f'  deg {deg_need}: {3*(deg_need+1)**2} 个 SH 系数，每高斯 {nf_need} floats')
print(f'  相对 deg 3 的内存 {nf_need/59:.2f}×  -> 100 万高斯 {nf_need*4*1e6/1e9:.3f} GB')
assert nf_need/59 > 4, '提高阶数的内存代价必须很大'
print(f'  ✓ 内存变 {nf_need/59:.1f} 倍，而这只是为了一种材质 —— 所以正确的做法是换表示')
print('    （GaussianShader / Relightable 3DGS：用显式 BRDF + 法向 + 环境光代替 SH）')

## 4 · 压缩账：两类手段可乘

In [ ]:
BASE_BYTES = 59*4

def bytes_per_gaussian(geo_bytes=4, sh_bytes=4, n_sh=48, extra=0):
    return GEO*geo_bytes + n_sh*sh_bytes + extra

print('① 改精度（量化）：')
print('  方案                        每高斯字节  相对基线   100万高斯')
schemes = [('全 fp32（基线）', 4, 4, 0), ('SH -> fp16', 4, 2, 0),
           ('SH -> int8 + 每高斯 scale', 4, 1, 4),
           ('几何 fp16 + SH int8', 2, 1, 4), ('全 int8 + scale', 1, 1, 8)]
for name, gb, sb, ex in schemes:
    v = bytes_per_gaussian(gb, sb, 48, ex)
    print(f'  {name:26s}  {v:5d} B    {v/BASE_BYTES:.2f}×    {v*1e6/1e9:.3f} GB')
assert bytes_per_gaussian(4, 4) == 236
assert bytes_per_gaussian(2, 1, 48, 4) == 74

print('\n② 改分配（按需给阶数，因为漫反射的高斯占绝大多数）：')
print('  降到 deg0 的比例   平均 floats   每高斯字节   相对基线   100万高斯')
for frac in [0.5, 0.8, 0.9, 0.95]:
    avg = frac*14 + (1-frac)*59
    print(f'      {frac:.0%}            {avg:5.1f}       {avg*4:5.0f} B     '
          f'{avg*4/BASE_BYTES:.2f}×    {avg*4*1e6/1e9:.3f} GB')

print('\n③ 两类可乘：95% 降 deg0 + SH int8 + 几何 fp16')
avg_sh = 0.95*3 + 0.05*48
combo = GEO*2 + avg_sh*1 + 4
print(f'  平均 SH 系数 {avg_sh:.2f} 个 -> {combo:.1f} B/高斯 = {combo/BASE_BYTES:.3f}×')
print(f'  100 万高斯只要 {combo*1e6/1e9:.3f} GB（基线 0.236 GB）')
assert combo/BASE_BYTES < 0.15, f'组合后应低于基线的 0.15×，实测 {combo/BASE_BYTES:.3f}'
print(f'  ✓ 压到基线的 {combo/BASE_BYTES:.1%} —— 这就是场景文件能从几百 MB 压到几十 MB 的原因')

# fp16 存位置的坑
print('\n⚠ 位置用 fp16 的坑：fp16 的相对精度约 %.1e' % np.finfo(np.float16).eps)
for extent in [1.0, 10.0, 100.0]:
    err = extent * float(np.finfo(np.float16).eps)
    print(f'  场景尺度 {extent:5.1f} m -> 远端位置误差约 {err*100:.2f} cm')
err100 = 100.0*float(np.finfo(np.float16).eps)
assert err100 > 0.05, 'fp16 在 100 m 场景里的误差必须超过 5 cm'
print(f'  ✓ 100 m 的场景里误差 {err100*100:.1f} cm，而高斯尺度常常只有几厘米')
print('    正确做法：分块量化（每格一个 fp32 原点 + 格内 int16 偏移），')
print('    精度随格子大小而不是场景大小')

## 5 · 浮物：构造一个「训练视角看不见、留出视角一眼看到」的例子

这是浮物最本质的形态。设置：训练视角集中在一个窄弧内（±5°），
留出视角在 40°。浮物放在**一个真实表面点的前面、且颜色与它相同** ——
于是训练视角下它完全无法被区分，换视角就露出来。

In [ ]:
# 2D 设置（一个 1D 图像）：相机绕原点转，看向一面 z=8 的墙
WALL_Z = 8.0
FOCAL = 200.0        # 半视场 atan(100/200) = 26.6°
NPIX = 200

def cam_dirs(angle_deg):
    '''相机在角度 angle_deg 处（绕 y 轴），返回 (相机位置, 每个像素的方向)。'''
    a = np.deg2rad(angle_deg)
    pos = np.array([-WALL_Z*np.sin(a), WALL_Z*(1-np.cos(a))])   # 绕墙心的弧上
    look = np.array([np.sin(a), np.cos(a)])                     # 看向墙心
    right = np.array([look[1], -look[0]])
    u = (np.arange(NPIX) - NPIX/2)/FOCAL
    d = look[None, :] + u[:, None]*right[None, :]
    return pos, d/np.linalg.norm(d, axis=1, keepdims=True)

def render_1d(angle_deg, blobs):
    '''blobs: [(中心xy, 半径, 颜色, alpha)]。返回 (颜色, 累积 A)。'''
    pos, dirs = cam_dirs(angle_deg)
    hits = []
    for c, r, col, al in blobs:
        oc = pos - np.asarray(c, float)
        b = dirs @ oc
        cc = oc @ oc - r*r
        disc = b*b - cc
        t = np.where(disc >= 0, -b - np.sqrt(np.maximum(disc, 0)), np.inf)
        t = np.where(t > 1e-6, t, np.inf)
        hits.append((t, col, al))
    order = np.argsort([np.nanmin(np.where(np.isfinite(h[0]), h[0], 1e9)) for h in hits])
    img = np.zeros(NPIX); A = np.zeros(NPIX); T = np.ones(NPIX)
    for i in order:
        t, col, al = hits[i]
        vis = np.isfinite(t)
        a = np.where(vis, al, 0.0)
        img += T*a*col; A += T*a; T *= (1-a)
    return img, A

# 场景：墙面由一串小球拼成（颜色 0.7），外加一个浮物
wall = [((x, WALL_Z), 0.22, 0.7, 0.98) for x in np.arange(-3.0, 3.01, 0.20)]
# 浮物：放在 (0, 4)，即墙前 4 m 的正中，颜色与墙相同
floater = [((0.0, 4.0), 0.30, 0.7, 0.9)]

TRAIN_ANGLES = [-15, -8, -3, 0, 3, 8, 15]
HOLD_ANGLES = [20, 25, 30]

print('训练视角（-15° ~ +15° 的弧内）下，有浮物 vs 无浮物：')
for ang in TRAIN_ANGLES:
    i_no, A_no = render_1d(ang, wall)
    i_yes, A_yes = render_1d(ang, wall + floater)
    print(f'  {ang:+3d}°: 最大像素差 {np.abs(i_yes-i_no).max():.4f}'
          f'   A 最大差 {np.abs(A_yes-A_no).max():.4f}'
          f'   受影响像素(>0.01) {(np.abs(i_yes-i_no) > 0.01).sum()}')

d_train = max(np.abs(render_1d(a, wall+floater)[0] - render_1d(a, wall)[0]).max()
              for a in TRAIN_ANGLES)
print(f'\n训练视角的最大图像差 {d_train:.4f}（= {d_train*255:.3f} 个 8bit 色阶）')

print('\n留出视角下：')
for ang in HOLD_ANGLES:
    i_no, A_no = render_1d(ang, wall)
    i_yes, A_yes = render_1d(ang, wall + floater)
    print(f'  {ang:+3d}°: 最大像素差 {np.abs(i_yes-i_no).max():.4f}'
          f'   A 最大差 {np.abs(A_yes-A_no).max():.4f}'
          f'   受影响像素 {(np.abs(i_yes-i_no) > 0.01).sum()}')

d_hold = max(np.abs(render_1d(a, wall+floater)[0] - render_1d(a, wall)[0]).max()
             for a in HOLD_ANGLES)
print(f'\n留出视角的最大图像差 {d_hold:.4f}，是训练视角的 '
      f'{d_hold/max(d_train,1e-12):.0f}×')
assert d_train < 0.01, f'训练视角下浮物应几乎不可见，实测 {d_train:.4f}'
assert d_hold > 0.30, f'留出视角下浮物应明显可见，实测 {d_hold:.4f}'
assert d_hold/d_train > 500, f'落差应超过 500 倍，实测 {d_hold/d_train:.0f}×'
print(f'\n✓ 落差 {d_hold/d_train:.0f} 倍。这就是浮物的本质：')
print(f'  它在训练损失上几乎免费（{d_train*255:.3f} 个色阶，远低于量化噪声），')
print(f'  在留出视角上一目了然（{d_hold*255:.0f} 个色阶）。')
print('  所以最便宜的检测法就是「从训练集之外的视角渲一张」。')
print()
print('  而它也说明为什么加正则救不了观测不足：')
print('  浮物在 -15°~+15° 这个弧内是**完全合法**的解 —— 训练数据不含否定它的信息。')
print('  任何正则都只是在猜一个先验，而不同的先验会猜出不同的、都能拟合训练视角的答案。')

i25_no, A25_no = render_1d(25, wall)
i25_yes, A25_yes = render_1d(25, wall + floater)
print('\n留出视角(25°)的累积不透明度 A 剖面（. <0.2  : <0.5  o <0.9  # 满）：')
for A, tag in [(A25_no, '无浮物'), (A25_yes, '有浮物')]:
    line = ''.join('.' if v < 0.2 else (':' if v < 0.5 else ('o' if v < 0.9 else '#'))
                   for v in A[::2])
    print(f'  {tag}: {line}')
# 浮物出现在「本该是背景」的地方
bg = A25_no < 0.2
assert bg.any(), '25° 视角下应该有一部分像素看到背景'
assert (A25_yes[bg] > 0.5).any(), '浮物应出现在本该是背景的像素上'
n_bad = int((A25_yes[bg] > 0.5).sum())
print(f'  ✓ 在 {int(bg.sum())} 个本该是背景（A<0.2）的像素里，'
      f'有 {n_bad} 个的 A 涨到 >0.5 —— 这是第二种检测法')

## 6 · 4DGS 的内存账

In [ ]:
def traj_floats(order, sh_deg=3, sh_static=True):
    '''多项式轨迹参数化下每个高斯的 floats。order=0 表示静态。'''
    nc = order + 1
    nsh = 3*(sh_deg+1)**2
    geo = (3 + 4 + 3 + 1) * nc          # mu, quat, scale, alpha 各 nc 个系数
    return geo + (nsh if sh_static else nsh*nc)

STATIC = floats_per_gaussian(3)
print(f'静态 3DGS: {STATIC} floats -> {STATIC*4*1e6/1e9:.3f} GB / 100 万高斯\n')
print('A) 逐帧独立：')
for T in [1, 10, 50, 300]:
    tag = '  (10 秒 @30fps)' if T == 300 else ''
    print(f'   {T:3d} 帧: {STATIC*T:6d} floats -> {STATIC*T*4*1e6/1e9:7.3f} GB{tag}')

print('\nB) 多项式轨迹（SH 静态）：')
for order in [1, 2, 3]:
    n = traj_floats(order)
    print(f'   {order} 阶: {n:3d} floats -> {n*4*1e6/1e9:.3f} GB')

n3 = traj_floats(3)
assert n3 == 92, f'3 阶应为 92 floats，实测 {n3}'
ratio = STATIC*300/n3
assert abs(ratio - 192) < 2, f'压缩比应约 192×，实测 {ratio:.0f}'
print(f'\n✓ 3 阶轨迹 {n3*4*1e6/1e9:.3f} GB vs 300 帧逐帧独立 '
      f'{STATIC*300*4*1e6/1e9:.1f} GB —— 省 {ratio:.0f}×')
print('  所以 4DGS 必须用参数化的时间模型，这是内存的硬约束而不是设计偏好')

print('\nC) 让 SH 也随时间变（很少有人做）：')
for order in [1, 3]:
    n = traj_floats(order, sh_static=False)
    print(f'   {order} 阶、SH 动态: {n:3d} floats -> {n*4*1e6/1e9:.3f} GB'
          f'   （是静态的 {n/STATIC:.1f}×）')
assert traj_floats(3, sh_static=False)/STATIC > 3.5
print('  ✓ 代价是静态的 4 倍，而收益（能表示光照随时间变化）通常不值 ——')
print('    所以几乎所有 4DGS 工作都假设「物体运动但外观不变」，')
print('    于是它们表示不了「灯被打开」「阴影扫过物体」')

print('\nD) 只让位置随时间变（最省，但表示不了旋转与形变）：')
for order in [1, 3]:
    n = 3*(order+1) + 4 + 3 + 1 + 48
    print(f'   {order} 阶: {n} floats -> {n*4*1e6/1e9:.3f} GB')

print('\n一个由内存账直接推出的建议：动态场景优先减**高斯数**，不是减轨迹阶数。')
print(f'  轨迹 3 阶 -> 1 阶只省 {1-traj_floats(1)/traj_floats(3):.0%}'
      f'（{traj_floats(3)} -> {traj_floats(1)} floats）')
print('  而高斯数减半省 50%')
assert 1 - traj_floats(1)/traj_floats(3) < 0.30, '降阶数的收益应小于减半高斯数'

---
## ✏️ 练习

四道题各自独立。先写 TODO，再跑下一格的自测。

### ✏️ 练习 1 · 球谐求值

实现 `my_sh_color(k, dirs, deg)`：给定系数 `k`（形状 `((deg+1)², 3)`）与方向 `dirs`（`(N,3)`），
返回 `(N,3)` 的颜色。按 3DGS 的约定：**结果加 0.5 再 clamp 到 [0,1]**。
基函数请用已有的 `sh_basis`。

In [ ]:
def my_sh_color(k, dirs, deg=3):
    '''返回 (N,3) 的颜色，已加 0.5 并 clamp 到 [0,1]。'''
    # TODO: Y = sh_basis(dirs, deg)  形状 (N, (deg+1)²)
    #       返回 clip(Y @ k + 0.5, 0, 1)
    raise NotImplementedError

In [ ]:
# ---- 自测 1 ----
_D = fibonacci_sphere(20_000)
_k0 = np.zeros((1, 3))
_k1 = np.zeros((4, 3)); _k1[2] = [0.8, 0.0, -0.8]          # Y_10 ∝ z
_k3 = np.zeros((16, 3)); _k3[0] = [0.5, 0.3, 0.1]; _k3[9] = [0.4, -0.4, 0.0]

# ① 系数全零 -> 处处 0.5 灰
_c = my_sh_color(_k0, _D, 0)
assert _c.shape == (len(_D), 3), f'形状 {_c.shape}'
assert np.allclose(_c, 0.5), '系数全零应给出 0.5 灰（不是黑）'

# ② deg 0 的颜色与方向无关
_kc = np.array([[1.0, 0.5, -0.5]])
_c0 = my_sh_color(_kc, _D, 0)
assert np.abs(_c0 - _c0[0]).max() < 1e-12, 'deg 0 必须各方向同色'
# 并且等于 C0*k + 0.5
assert np.allclose(_c0[0], np.clip(C0*_kc[0] + 0.5, 0, 1)), 'deg0 的闭式核对失败'

# ③ deg 1 的 Y_10 造成沿 z 的梯度
_c1 = my_sh_color(_k1, _D, 1)
_up = _c1[_D[:, 2] > 0.9].mean(0); _dn = _c1[_D[:, 2] < -0.9].mean(0)
assert _up[0] > _dn[0] + 0.1 and _up[2] < _dn[2] - 0.1, 'Y_10 应造成沿 z 的反向梯度'

# ④ 输出必须被 clamp
_kbig = np.zeros((16, 3)); _kbig[0] = [100.0, -100.0, 0.0]
_cb = my_sh_color(_kbig, _D, 3)
assert _cb.min() >= 0.0 and _cb.max() <= 1.0, '必须 clamp 到 [0,1]'
assert np.allclose(_cb[:, 0], 1.0) and np.allclose(_cb[:, 1], 0.0), 'clamp 必须生效'

# ⑤ 与直接用 sh_basis 一致，且方向无需预先归一化
_c3 = my_sh_color(_k3, _D, 3)
assert np.allclose(_c3, np.clip(sh_basis(_D, 3) @ _k3 + 0.5, 0, 1))
_D2 = _D * 3.7                                  # 未归一化
assert np.allclose(my_sh_color(_k3, _D2, 3), _c3, atol=1e-12), '方向应内部归一化'

# ⑥ 均值等于 deg0 项（因为高阶基在球面上积分为 0）
_mean = _c3.mean(0)
_expect = np.clip(C0*_k3[0] + 0.5, 0, 1)
assert np.abs(_mean - _expect).max() < 0.01, \
    f'球面均值应等于 deg0 项：{np.round(_mean,4)} vs {np.round(_expect,4)}'
print(f'✓ 练习 1 通过：全零 -> 0.5 灰；deg0 各方向同色；Y_10 的梯度；'
      f'clamp 生效；球面均值 {np.round(_mean,4)} = deg0 项 {np.round(_expect,4)}')

### 📖 参考答案 1

In [ ]:
def my_sh_color(k, dirs, deg=3):
    Y = sh_basis(dirs, deg)
    return np.clip(Y @ np.asarray(k, float) + 0.5, 0.0, 1.0)

print('参考答案 1 已定义')
print('要点一：那个 +0.5 不是装饰 —— 它意味着「所有系数为 0」对应中性灰而不是黑，')
print('       所以初始化时把 SH 系数置零得到的是一个灰场景，优化从那里出发。')
print('要点二：⑥ 那条（球面均值 = deg0 项）是球谐正交性的直接后果，')
print('       也是「低阶系数决定整体亮度、高阶只是修饰」这句话的精确版本 ——')
print('       这正是官方每 1000 步才把阶数 +1 的理由：先定大局再修细节。')
print('要点三：clamp 让梯度在饱和处为零。一个过曝的高斯会「卡住」不再被优化，')
print('       这是一个真实的失败模式（表现为局部的死白斑块）。')

### ✏️ 练习 2 · 每个阶数能表示的最窄波瓣

实现 `my_narrowest(deg, tol)`：二分求出 degree ≤ `deg` 的带限球谐
能把 $\max(\cos\theta,0)^n$ 拟合到相对 L2 残差 `tol` 以内的**最大** $n$，
返回它对应的半强度半角（度）。

残差与半角请用已有的 `zonal_fit_residual` 与 `half_angle`。

In [ ]:
def my_narrowest(deg, tol=0.20):
    '''返回该阶数能表示的最窄波瓣的半强度半角（度）。'''
    # TODO: 在 n ∈ [1, 5000] 上二分。注意 n 越大波瓣越窄、残差越大。
    #       若连 n=1 都拟合不了（残差 >= tol），直接返回 half_angle(1)
    #       二分 60 次后返回 half_angle(lo)
    raise NotImplementedError

In [ ]:
# ---- 自测 2 ----
_m = {d: my_narrowest(d, 0.20) for d in [0, 1, 2, 3, 4, 8]}
print('  deg  实测最窄半角   125/(deg+1)   180/(deg+1)')
for d, v in _m.items():
    print(f'   {d}      {v:6.1f}°       {125/(d+1):6.1f}°      {180/(d+1):6.1f}°')

# ① 与参考实现一致
for d in [2, 3, 4]:
    assert abs(my_narrowest(d, 0.20) - narrowest_lobe(d, 0.20)) < 1e-6

# ② 阶数越高能表示越窄的波瓣（单调）
_vals = [_m[d] for d in [1, 2, 3, 4, 8]]
assert all(_vals[i] > _vals[i+1] for i in range(len(_vals)-1)), \
    f'必须单调递减，实测 {[f"{v:.1f}" for v in _vals]}'

# ③ deg 3 应为 31.2°
assert abs(_m[3] - 31.2) < 0.5, f'deg 3 应为 31.2°，实测 {_m[3]:.1f}°'

# ④ 系统性地低于 180/(deg+1)，且接近 125/(deg+1)
for d in [2, 3, 4, 8]:
    assert _m[d] < 180/(d+1), f'deg {d}: 实测界必须低于 180/(deg+1)'
    assert abs(_m[d] - 125/(d+1)) < 3.0, \
        f'deg {d}: 实测 {_m[d]:.1f}° 应接近 {125/(d+1):.1f}°'
_ratios = [_m[d]/(180/(d+1)) for d in [2, 3, 4, 8]]
assert 0.6 < min(_ratios) and max(_ratios) < 0.8, \
    f'实测/说法 应稳定在 0.6~0.8，实测 {[f"{r:.2f}" for r in _ratios]}'

# ⑤ 判据放宽时能表示更窄的波瓣
assert my_narrowest(3, 0.40) < my_narrowest(3, 0.20) < my_narrowest(3, 0.05), \
    'tol 越大，能「表示」的波瓣越窄'
# ⑥ 真实材质的对照：抛光塑料（12.3°）超出 deg 3 的能力
assert _m[3] > half_angle(30), \
    f'deg3 的界 {_m[3]:.1f}° 必须宽于抛光塑料的 {half_angle(30):.1f}°（即表示不了）'
print(f'\n✓ 练习 2 通过：deg3 的界 {_m[3]:.1f}°，'
      f'实测/说法 = {np.mean(_ratios):.2f}；'
      f'抛光塑料需 {half_angle(30):.1f}° —— 表示不了')

### 📖 参考答案 2

In [ ]:
def my_narrowest(deg, tol=0.20):
    lo, hi = 1.0, 5000.0
    if zonal_fit_residual(lo, deg) >= tol:
        return half_angle(lo)
    for _ in range(60):
        mid = (lo + hi)/2
        if zonal_fit_residual(mid, deg) < tol:
            lo = mid
        else:
            hi = mid
    return half_angle(lo)

print('参考答案 2 已定义')
print('要点一：二分的单调性前提是「n 越大残差越大」。这在整体上成立，')
print('       但**局部不成立** —— deg3 在 n=2 处的残差(0.029)低于 n=1(0.088)。')
print('       所以二分找到的是「上界的一个保守估计」，而不是严格的分界点。')
print('       我把这一点写出来，因为它是这类二分最常见的隐含假设错误。')
print('要点二：结论是 125/(deg+1) 而不是流行的 180/(deg+1) ——')
print('       后者高估了 SH 的能力约 1.45 倍。')
print('要点三：常数 125 依赖「20% 残差」这个判据（自测 ⑤ 验证了这一点）。')
print('       报一个经验常数时必须同时报判据，否则它不可复现。')

### ✏️ 练习 3 · 压缩账

实现 `my_bytes(deg0_frac, geo_bytes, sh_bytes, extra)`：
把 `deg0_frac` 比例的高斯降到 deg 0（3 个 SH 系数）、其余保持 deg 3（48 个），
几何 11 个数每个 `geo_bytes` 字节、SH 每个 `sh_bytes` 字节，
每高斯再加 `extra` 字节（量化的 scale）。返回**平均**每高斯字节数。

In [ ]:
def my_bytes(deg0_frac=0.0, geo_bytes=4, sh_bytes=4, extra=0):
    '''平均每高斯字节数。'''
    # TODO: 平均 SH 系数个数 = deg0_frac*3 + (1-deg0_frac)*48
    #       返回 11*geo_bytes + 平均SH系数*sh_bytes + extra
    raise NotImplementedError

In [ ]:
# ---- 自测 3 ----
_BASE = 59*4
# ① 基线
assert abs(my_bytes() - 236) < 1e-9, f'基线应为 236 B，实测 {my_bytes()}'
# ② SH -> fp16
assert abs(my_bytes(sh_bytes=2) - 140) < 1e-9
# ③ 几何 fp16 + SH int8 + 4 字节 scale
assert abs(my_bytes(geo_bytes=2, sh_bytes=1, extra=4) - 74) < 1e-9
# ④ 全部降到 deg0 时，SH 只剩 3 个系数
assert abs(my_bytes(deg0_frac=1.0) - (11*4 + 3*4)) < 1e-9, '全 deg0 应为 56 B'
# ⑤ 单调性
assert my_bytes(0.0) > my_bytes(0.5) > my_bytes(0.9) > my_bytes(1.0)
assert my_bytes(sh_bytes=4) > my_bytes(sh_bytes=2) > my_bytes(sh_bytes=1)
# ⑥ 两类手段可乘：组合应低于任一单独手段
_only_quant = my_bytes(0.0, 2, 1, 4)
_only_alloc = my_bytes(0.95, 4, 4, 0)
_combo = my_bytes(0.95, 2, 1, 4)
assert _combo < _only_quant and _combo < _only_alloc, '组合必须优于任一单独手段'
assert _combo/_BASE < 0.15, f'组合应低于基线的 0.15×，实测 {_combo/_BASE:.3f}'
# ⑦ 与讲解页的具体数字一致
assert abs(my_bytes(0.9) - 74) < 1e-9, '90% 降 deg0 应为 74 B'
assert abs(_combo - 31.25) < 0.1, f'组合应约 31.2 B，实测 {_combo:.2f}'
print(f'✓ 练习 3 通过：基线 {my_bytes():.0f} B；'
      f'只量化 {_only_quant:.0f} B ({_only_quant/_BASE:.2f}×)；'
      f'只改分配 {_only_alloc:.0f} B ({_only_alloc/_BASE:.2f}×)；'
      f'组合 {_combo:.1f} B ({_combo/_BASE:.3f}×)')

### 📖 参考答案 3

In [ ]:
def my_bytes(deg0_frac=0.0, geo_bytes=4, sh_bytes=4, extra=0):
    avg_sh = deg0_frac*3 + (1.0 - deg0_frac)*48
    return 11*geo_bytes + avg_sh*sh_bytes + extra

print('参考答案 3 已定义')
print('要点一：⑥ 那条（两类手段可乘）是压缩工作的基本结构 ——')
print('       量化改的是「每个数占几个字节」，按需分配改的是「有多少个数」。')
print('要点二：但**顺序**很重要。正确的顺序是：')
print('       先剪高斯（模块 04：同时降存储、排序与渲染成本）')
print('       -> 再按需分配 SH 阶数 -> 最后量化。')
print('       反过来做，你会花很大力气去量化本该被删掉的高斯。')
print('要点三：这个模型没算上「高斯数」这一维，而那是最有效的一维 ——')
print('       所以别把这个函数当成压缩比的完整预测。')

### ✏️ 练习 4 · 4DGS 的内存账

实现 `my_traj(order, sh_deg, sh_static)`：多项式轨迹参数化下每个高斯的 floats。
几何量（$\mu$ 3 个、$q$ 4 个、$s$ 3 个、$\alpha$ 1 个，共 11 个）
各有 `order+1` 个多项式系数；SH 在 `sh_static=True` 时不随时间变。

In [ ]:
def my_traj(order, sh_deg=3, sh_static=True):
    '''多项式轨迹参数化下每高斯的 floats。'''
    # TODO: nc = order+1；nsh = 3*(sh_deg+1)²
    #       几何 = 11*nc；SH = nsh（静态）或 nsh*nc（动态）
    raise NotImplementedError

In [ ]:
# ---- 自测 4 ----
# ① order=0 应退化为静态 3DGS
assert my_traj(0) == 59, f'order=0 应为 59 floats，实测 {my_traj(0)}'
# ② 讲解页的三个数
assert my_traj(1) == 70 and my_traj(2) == 81 and my_traj(3) == 92, \
    f'1/2/3 阶应为 70/81/92，实测 {my_traj(1)}/{my_traj(2)}/{my_traj(3)}'
# ③ 与参考实现一致
for _o in [0, 1, 2, 3, 5]:
    for _st in [True, False]:
        assert my_traj(_o, 3, _st) == traj_floats(_o, 3, _st)
# ④ 每加一阶固定加 11 个 float（SH 静态时）
_d = [my_traj(o+1) - my_traj(o) for o in range(5)]
assert all(v == 11 for v in _d), f'SH 静态时每阶应固定 +11，实测 {_d}'
# ⑤ SH 动态的代价
_dyn3 = my_traj(3, 3, sh_static=False)
assert _dyn3 == 11*4 + 48*4, f'3 阶 SH 动态应为 {11*4+48*4}，实测 {_dyn3}'
assert _dyn3/59 > 3.5, f'SH 动态应是静态的 3.5 倍以上，实测 {_dyn3/59:.1f}×'
# ⑥ 压缩比：3 阶轨迹 vs 300 帧逐帧独立
_ratio = 59*300/my_traj(3)
assert abs(_ratio - 192) < 2, f'压缩比应约 192×，实测 {_ratio:.0f}'
# ⑦ 降阶数的收益远小于减半高斯数
_gain_order = 1 - my_traj(1)/my_traj(3)
assert _gain_order < 0.30, f'3 阶 -> 1 阶只省 {_gain_order:.0%}，应小于 30%'
assert _gain_order < 0.5, '必须小于「高斯数减半」的 50%'
# ⑧ 低阶 SH 时几何占比上升
assert my_traj(3, 0) == 11*4 + 3, f'sh_deg=0 时 3 阶应为 {11*4+3}'
assert (11*4)/my_traj(3, 0) > (11*4)/my_traj(3, 3), 'SH 阶数低时几何占比更高'
print(f'✓ 练习 4 通过：0/1/2/3 阶 = {my_traj(0)}/{my_traj(1)}/{my_traj(2)}/{my_traj(3)} floats；'
      f'压缩比 {_ratio:.0f}×；降阶数只省 {_gain_order:.0%}（减半高斯数省 50%）')

### 📖 参考答案 4

In [ ]:
def my_traj(order, sh_deg=3, sh_static=True):
    nc = order + 1
    nsh = 3*(sh_deg+1)**2
    geo = (3 + 4 + 3 + 1) * nc
    return geo + (nsh if sh_static else nsh*nc)

print('参考答案 4 已定义')
print('要点一：⑦ 是本节最有用的一条 —— 轨迹从 3 阶降到 1 阶只省 24%，')
print('       而高斯数减半省 50%。所以动态场景优先减高斯数。')
print('要点二：⑤ 说明为什么几乎所有 4DGS 工作都让 SH 静态：动态 SH 是 4 倍代价。')
print('       代价换来的能力是「表示光照随时间变化」，而这通常不是需求。')
print('       后果：4DGS 表示不了「灯被打开」「阴影扫过物体」—— 这是一条硬边界。')
print('要点三：这个账也解释了为什么 4DGS 的论文里高斯数常常比静态 3DGS **少** ——')
print('       不是它更高效，而是它必须省着用。')

---
## 🧪 真实工程胶囊

```python
# ---- 官方实现：球谐（utils/sh_utils.py + cuda_rasterizer/forward.cu）----
# Python 侧只做「阶数渐进提升」：
class GaussianModel:
    def oneupSHdegree(self):
        if self.active_sh_degree < self.max_sh_degree:
            self.active_sh_degree += 1
# train.py: 每 1000 步调一次 —— 先定大局（低阶）再修细节（高阶）
if iteration % 1000 == 0:
    gaussians.oneupSHdegree()

# CUDA 侧的求值（forward.cu, computeColorFromSH）：
#   glm::vec3 dir = pos - campos;  dir = dir / glm::length(dir);   // 相机 -> 高斯
#   glm::vec3 result = SH_C0 * sh[0];
#   if (deg > 0) { result = result - SH_C1*y*sh[1] + SH_C1*z*sh[2] - SH_C1*x*sh[3]; ... }
#   result += 0.5f;                     // ← 练习 1 的那个 +0.5
#   return glm::max(result, 0.0f);      // clamp -> 饱和处梯度为零

# 两组系数是分开存的（便于只优化 dc 或只优化 rest）：
#   self._features_dc   : (N, 1, 3)      deg 0 的那 3 个
#   self._features_rest : (N, 15, 3)     deg 1-3 的那 45 个
# get_features 把它们 cat 起来 -> (N, 16, 3)

# ---- 压缩：几个能直接用的工具 ----
# 1) SOG / .spz 交换格式（Niantic 的 spz，约 10× 无感压缩）
#    pip install spz  ->  spz.save(gaussians, "scene.spz")
# 2) LightGaussian：SH 蒸馏到低阶 + 剪枝 + 量化
# 3) gsplat 自带的 PNG-based 压缩
from gsplat.compression import PngCompression
PngCompression().compress("out_dir", splats)     # 位置/SH 分别量化后存成 PNG

# ---- 4DGS：两个主流实现 ----
# Deformable 3DGS: 一个小 MLP 把 (xyz, t) 映到 (Δxyz, Δquat, Δscale)
# 4DGS (HexPlane): 用 6 个 2D 特征平面编码 4D 时空，再解码成形变
# 共同点：SH **静态** —— 所以都表示不了光照随时间变化（练习 4 ⑤）

# ---- 想要重打光的话 ----
# GaussianShader / Relightable 3DGS：把 SH 换成
#   (albedo, roughness, metallic, normal) + 一个环境光表示
# 代价：引入逆渲染的全部歧义，PSNR 通常反而下降
```

**排查清单**

| 症状 | 先查什么 | 依据 |
|---|---|---|
| 金属/玻璃看起来像哑光 | `sh_degree`；但**提高阶数救不了** | deg 3 只到半角 31.2°，抛光塑料要 12.3° |
| 局部有死白斑块，训练不动 | SH 求值后的 clamp | 饱和处梯度为零，高斯会「卡住」 |
| 换光照/搬到别的场景就崩 | 这是设计边界，不是 bug | 光照被烘进 SH，没有材质分解 |
| 场景文件几百 MB | 先剪高斯，再降 SH 阶，最后量化 | deg 3 时 SH 占 81% |
| 位置量化后远处物体错位 | 是否用了全局 fp16 | 100 m 场景里 fp16 误差 9.8 cm |
| 换视角就出现半透明色块 | 浮物；从留出视角渲一张确认 | 训练视角差 0.0003，留出视角差 0.6300（2100×） |
| 动态场景 OOM | 减**高斯数**，不是减轨迹阶数 | 3 阶→1 阶只省 24%，减半高斯省 50% |
| 4DGS 表示不了灯被打开 | 这是设计边界 | SH 静态；动态 SH 是 4 倍内存 |